In [ ]:
library(dplyr)
library(tidyr)

data <- read.csv("Data.csv")

# Rename first column to ID consistently
names(data)[1] <- "ID"

# ── Universal variables ───────────────────────────────────────────────────────
total_lamp_height    <- 358.14 + 20.9
stick_fixation_point <- total_lamp_height
outer_planter_height <- 40
planter_height       <- 20.9
BlueBall_height      <- outer_planter_height + 11.3025
observer_height      <- 118.8

# ── Helpers ───────────────────────────────────────────────────────────────────
`%||%` <- function(a, b) if (!is.null(a)) a else b

calc_va <- function(size, dist) {
  ifelse(is.na(size) | is.na(dist) | dist == 0,
         NA_real_,
         (2 * atan(size / (2 * dist))) * (180 / pi))
}

get_object_height <- function(obj) {
  if (grepl("^Stick", obj)) {
    total_lamp_height
  } else if (grepl("^Lamp", obj)) {
    total_lamp_height + 20.32
  } else if (obj == "BlueBall") {
    BlueBall_height
  } else if (grepl("^MoonBall", obj)) {
    planter_height + 27.5
  } else {
    NA_real_
  }
}

get_size_col <- function(obj) {
  if (grepl("^Stick", obj)) paste0(obj, ".Height..cm.")
  else                       paste0(obj, ".Width..cm.")
}

# ── Detect objects flexibly ────────────────────────────────────────────────────
objects <- unique(sub("\\.(Estimate|Adjusted)\\..*", "",
                      names(data)[grepl("\\.(Estimate|Adjusted)\\.", names(data))]))

# ── Compute per-object derived columns ────────────────────────────────────────
for (obj in objects) {

  dist_col <- paste0(obj, ".Distance..cm.")
  size_col <- get_size_col(obj)
  est_size <- paste0(obj, ".Estimate.Template.Size..cm.")
  est_dist <- paste0(obj, ".Estimate.Distance..cm.")
  adj_size <- paste0(obj, ".Adjusted.Template.Size..cm.")
  adj_dist <- paste0(obj, ".Adjusted.Distance..cm.")

  obj_h <- get_object_height(obj)

  dist <- if (dist_col %in% names(data)) data[[dist_col]] else NA
  size <- if (size_col %in% names(data)) data[[size_col]] else NA

  # Real VA + elevation
  if (dist_col %in% names(data)) {
    data[[paste0(obj, "_Real_VA")]] <- calc_va(size, dist)
    data[[paste0(obj, "_Elevation")]] <- atan(obj_h / dist) * (180 / pi)
  }

  # Perceptual VA
  if (all(c(est_size, est_dist) %in% names(data))) {
    data[[paste0(obj, "_Perceptual_VA")]] <-
      calc_va(data[[est_size]], data[[est_dist]])
  }

  # Adjusted VA
  if (all(c(adj_size, adj_dist) %in% names(data))) {
    data[[paste0(obj, "_Adjusted_VA")]] <-
      calc_va(data[[adj_size]], data[[adj_dist]])
  }
}

# ── Pivot to long format ──────────────────────────────────────────────────────
va_cols <- names(data)[grepl("_(Perceptual|Adjusted)_VA$", names(data))]

data_long <- data %>%
  select(any_of(c("ID","Gender","Date","Time","Age","Version")),
         all_of(va_cols)) %>%
  pivot_longer(
    cols = all_of(va_cols),
    names_to = "Measurement",
    values_to = "Reported_Visual_Angle"
  ) %>%
  filter(!is.na(Reported_Visual_Angle)) %>%
  mutate(
    Object = sub("_(Perceptual|Adjusted)_VA$", "", Measurement),
    Task   = if_else(grepl("Perceptual", Measurement), "Perceptual", "Adjusted")
  )

# ── Object-level lookup ───────────────────────────────────────────────────────
object_lookup <- bind_rows(lapply(objects, function(obj) {

  dist_col  <- paste0(obj, ".Distance..cm.")
  width_col <- paste0(obj, ".Width..cm.")

  if (!dist_col %in% names(data)) return(NULL)

  obj_h <- get_object_height(obj)

  data %>%
    transmute(
      ID,
      Object = obj,
      Ground_Distance   = .data[[dist_col]],
      Real_Visual_Angle = if (paste0(obj, "_Real_VA") %in% names(data))
                            .data[[paste0(obj, "_Real_VA")]] else NA_real_,
      Ground_Elevation  = if (paste0(obj, "_Elevation") %in% names(data))
                            .data[[paste0(obj, "_Elevation")]] else NA_real_,
      Width             = if (width_col %in% names(data))
                            .data[[width_col]] else NA_real_,
      Observer_Distance = sqrt(Ground_Distance^2 + (obj_h - observer_height)^2),
      Observer_Elevation = atan((obj_h - observer_height) / Ground_Distance) * 180 / pi
    )
}))

# ── Task-level lookup ─────────────────────────────────────────────────────────
task_lookup <- bind_rows(lapply(objects, function(obj) {

  out <- list()

  est_size <- paste0(obj, ".Estimate.Template.Size..cm.")
  est_dist <- paste0(obj, ".Estimate.Distance..cm.")
  adj_size <- paste0(obj, ".Adjusted.Template.Size..cm.")
  adj_dist <- paste0(obj, ".Adjusted.Distance..cm.")

  if (all(c(est_size, est_dist) %in% names(data))) {
    out[[length(out)+1]] <- data %>%
      transmute(
        ID,
        Object = obj,
        Task = "Perceptual",
        Template_Size = .data[[est_size]],
        Arm_Length    = .data[[est_dist]]
      )
  }

  if (all(c(adj_size, adj_dist) %in% names(data))) {
    out[[length(out)+1]] <- data %>%
      transmute(
        ID,
        Object = obj,
        Task = "Adjusted",
        Template_Size = .data[[adj_size]],
        Arm_Length    = .data[[adj_dist]]
      )
  }

  bind_rows(out)
}))

# ── Final join ────────────────────────────────────────────────────────────────
data_long <- data_long %>%
  left_join(object_lookup, by = c("ID", "Object")) %>%
  left_join(task_lookup,   by = c("ID", "Object", "Task")) %>%
  mutate(
    Ratio_Visual_Angle = if_else(
      is.na(Real_Visual_Angle) | Real_Visual_Angle == 0,
      NA_real_,
      Reported_Visual_Angle / Real_Visual_Angle
    )
  )

write.csv(data_long, file = "AllQuadDataLong.csv", row.names = FALSE)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


